This notebook demonstrates simple `.tlspec` round-trip + the new option-class style for capture/save.

In [ ]:
import torch
import torchlens as tl
from _shared import tiny_model

In [ ]:
# COVERS: tl.options.CaptureOptions, tl.options.SaveOptions, ModelLog.save (placeholder)
x = torch.randn(1, 4)
model = tiny_model()
log = tl.log_forward_pass(model, x, capture=tl.options.CaptureOptions(layers_to_save=["fc1"]))

In [ ]:
# COVERS: ModelLog.first_nonfinite, NaN/Inf save-load preservation
from tempfile import TemporaryDirectory


class NonFiniteFixture(torch.nn.Module):
    def forward(self, x):
        zeros = x - x
        return zeros / zeros


nan_log = tl.log_forward_pass(NonFiniteFixture(), torch.ones(1, 2))
assert "First non-finite" in nan_log.first_nonfinite()
assert "NaN/Inf" in str(nan_log)
with TemporaryDirectory() as tmpdir:
    path = f"{tmpdir}/nan_case.tlspec"
    tl.save(nan_log, path, overwrite=True)
    loaded = tl.load(path)
assert "First non-finite" in loaded.first_nonfinite()